### Configuración inicial

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
from delta import *
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, TimestampType


builder = SparkSession.builder \
    .appName("Lab_SECOP_Silver") \
    .master("local[*]") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/01/30 22:30:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### Leer datos de Bronce

In [11]:
bronze_path = "/app/data/lakehouse/bronze/secop"
df_bronze= spark.read.format("delta").load(bronze_path)

In [12]:
df_bronze.limit(5).toPandas()

,anno_bpin,c_digo_bpin,ciudad,codigo_de_categoria_principal,codigo_entidad,codigo_proveedor,condiciones_de_entrega,departamento,descripcion_del_proceso,descripcion_documentos_tipo,...,valor_amortizado,valor_de_pago_adelantado,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de,valor_pendiente_de_ejecucion,valor_pendiente_de_pago,_ingestion_time,_source_file
0,2025,202500000002779,Chinácota,V1.80111600,704851104,730599727,No Definido,Norte de Santander,PRESTAR LOS SERVICIOS PROFESIONALES COMO FORMA...,No definido,...,0,0,12343333,12343333,11500000,0,843333,843333,2026-01-30 22:18:40.643078,API_Socrata_Bogota_2025
1,No D,No Definido,No Definido,V1.80111607,705008498,718547920,Como acordado previamente,No Definido,Brindar acompañamiento jurídico y apoyo profes...,No definido,...,0,0,20000000,20000000,20000000,0,0,0,2026-01-30 22:18:40.643078,API_Socrata_Bogota_2025
2,No D,No Definido,Cartagena,V1.85121600,709192637,712389626,No Definido,Bolívar,PRESTACIÓN DE LOS SERVICIOS COMO GESTORES DE S...,No definido,...,0,0,4000000,4000000,4000000,0,0,0,2026-01-30 22:18:40.643078,API_Socrata_Bogota_2025
3,No D,No Definido,Los Patios,V1.85101601,713088169,731045159,No Definido,Norte de Santander,PRESTAR SUS SERVICIOS EN CONDICIÓN DE TÉCNICO ...,No definido,...,0,0,10000000,10000000,10000000,0,0,0,2026-01-30 22:18:40.643078,API_Socrata_Bogota_2025
4,No D,No Definido,Amalfi,V1.81111819,718317027,718871106,A convenir,Antioquia,Prestación de servicios de Asesoría en Calidad...,No definido,...,0,0,12637800,8425200,8425200,0,4212600,4212600,2026-01-30 22:18:40.643078,API_Socrata_Bogota_2025


### Transformación del tipo de dato

In [16]:
# grupos de columnas 

# Números (Money/Decimal)
cols_decimal = [
    "valor_del_contrato", "valor_de_pago_adelantado", "valor_facturado", 
    "valor_pendiente_de_pago", "valor_pagado", "valor_amortizado", 
    "valor_pendiente_de", "valor_pendiente_de_ejecucion", "saldo_cdp", 
    "saldo_vigencia", "presupuesto_general_de_la_nacion_pgn", 
    "sistema_general_de_participaciones", "sistema_general_de_regal_as",
    "recursos_propios_alcald_as_gobernaciones_y_resguardos_ind_genas_",
    "recursos_de_credito", "recursos_propios"
]

#  Fechas (Timestamp)
cols_timestamp = [
    "fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato",
    "ultima_actualizacion", "fecha_inicio_liquidacion", "fecha_fin_liquidacion",
    "fecha_de_notificaci_n_de_prorrogaci_n"
]

# números enteros o IDs
cols_int = ["nit_entidad", "codigo_entidad", "dias_adicionados"]

# 3. Aplicar las transformaciones
df_silver_typed = df_bronze

for c in df_bronze.columns:
    if c in cols_decimal:
        df_silver_typed = df_silver_typed.withColumn(c, F.col(c).cast(DecimalType(18, 2)))
    elif c in cols_timestamp:
        df_silver_typed = df_silver_typed.withColumn(c, F.col(c).cast("timestamp"))
    elif c in cols_int:
        df_silver_typed = df_silver_typed.withColumn(c, F.col(c).cast("long"))
    # El resto se quedan como String automáticamente

print("Tipado completo. Revisando esquema...")
df_silver_typed.printSchema()

Tipado completo. Revisando esquema...
root
 |-- anno_bpin: string (nullable = true)
 |-- c_digo_bpin: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- codigo_de_categoria_principal: string (nullable = true)
 |-- codigo_entidad: long (nullable = true)
 |-- codigo_proveedor: string (nullable = true)
 |-- condiciones_de_entrega: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- descripcion_del_proceso: string (nullable = true)
 |-- descripcion_documentos_tipo: string (nullable = true)
 |-- destino_gasto: string (nullable = true)
 |-- dias_adicionados: long (nullable = true)
 |-- documento_proveedor: string (nullable = true)
 |-- documentos_tipo: string (nullable = true)
 |-- domicilio_representante_legal: string (nullable = true)
 |-- duraci_n_del_contrato: string (nullable = true)
 |-- el_contrato_puede_ser_prorrogado: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- es_grupo: string (nullable = true)
 |-- es_pym

In [22]:
def resumen_robusto_numericas(df, columnas):
    resultados = []

    for col in columnas:
        # 1. Calculamos las estadísticas
        stats = df.select(
            F.count(F.col(col)).alias("n"),
            F.mean(F.when(F.col(col).isNull(), 1).otherwise(0)).alias("missing_prop"),
            F.expr(f"percentile_approx({col}, 0.5)").alias("mediana"),
            F.expr(f"percentile_approx({col}, 0.1)").alias("p10"),
            F.expr(f"percentile_approx({col}, 0.9)").alias("p90"),
            (
                F.expr(f"percentile_approx({col}, 0.75)") -
                F.expr(f"percentile_approx({col}, 0.25)")
            ).alias("iqr"),
            F.mean(F.when(F.col(col) == 0, 1).otherwise(0)).alias("ceros_prop")
        ).collect()[0]

        # 2. Convertimos TODO a float para evitar el error de mezcla de tipos
        # Usamos float() para asegurar que Decimal y Long convivan en paz
        resultados.append((
            col,
            int(stats["n"]),
            round(float(stats["missing_prop"]) * 100, 2),
            float(stats["mediana"]) if stats["mediana"] is not None else 0.0,
            float(stats["p10"]) if stats["p10"] is not None else 0.0,
            float(stats["p90"]) if stats["p90"] is not None else 0.0,
            float(stats["iqr"]) if stats["iqr"] is not None else 0.0,
            round(float(stats["ceros_prop"]) * 100, 2)
        ))

    return spark.createDataFrame(
        resultados,
        [
            "variable", "n", "missing_%", "mediana", "p10", "p90", "iqr", "ceros_%"
        ]
    )

In [23]:
# Asegúrate de que vars_numericas tenga los nombres que están en tu df_silver_typed
resumen_num = resumen_robusto_numericas(df_silver_typed, vars_numericas)
resumen_num.show()

+------------------+------+---------+-----------+---------+------------+---------+-------+
|          variable|     n|missing_%|    mediana|      p10|         p90|      iqr|ceros_%|
+------------------+------+---------+-----------+---------+------------+---------+-------+
|valor_del_contrato|456617|      0.0|1.4227895E7|4000000.0|1.17311116E8|   2.45E7|   0.85|
|      valor_pagado|456617|      0.0|        0.0|      0.0|   1.63956E7|5962500.0|  66.74|
|  dias_adicionados|456617|      0.0|        0.0|      0.0|        16.0|      0.0|  86.55|
+------------------+------+---------+-----------+---------+------------+---------+-------+



In [26]:

# ANÁLISIS DE CATEGÓRICAS

print(" Generando descriptivas categóricas...")

# Definicion de variables
vars_categoricas = [
    "tipo_de_contrato", 
    "modalidad_de_contratacion", 
    "estado_contrato",
    "sector",
    "orden"
]

resumen_cat = resumen_categoricas(df_silver_typed, vars_categoricas)

# Imprimir los resultados 
for columna, tabla_frecuencia in resumen_cat.items():
    print(f"\n Top 5 para la columna: {columna.upper()}")
    tabla_frecuencia.show(truncate=False)

 Generando descriptivas categóricas...



 Top 5 para la columna: TIPO_DE_CONTRATO


+-----------------------+------+----------+
|tipo_de_contrato       |count |porcentaje|
+-----------------------+------+----------+
|Prestación de servicios|359310|78.69     |
|Otro                   |26529 |5.81      |
|Decreto 092 de 2017    |24869 |5.45      |
|Compraventa            |14466 |3.17      |
|Suministros            |14406 |3.15      |
+-----------------------+------+----------+


 Top 5 para la columna: MODALIDAD_DE_CONTRATACION


+------------------------------------+------+----------+
|modalidad_de_contratacion           |count |porcentaje|
+------------------------------------+------+----------+
|Contratación directa                |313201|68.59     |
|Contratación régimen especial       |76965 |16.86     |
|Mínima cuantía                      |32876 |7.2       |
|Selección Abreviada de Menor Cuantía|7677  |1.68      |
|Selección abreviada subasta inversa |7623  |1.67      |
+------------------------------------+------+----------+


 Top 5 para la columna: ESTADO_CONTRATO
+---------------+------+----------+
|estado_contrato|count |porcentaje|
+---------------+------+----------+
|En ejecución   |308634|67.59     |
|Modificado     |97220 |21.29     |
|terminado      |23144 |5.07      |
|Aprobado       |12849 |2.81      |
|Cerrado        |10169 |2.23      |
+---------------+------+----------+


 Top 5 para la columna: SECTOR


+-------------------------+------+----------+
|sector                   |count |porcentaje|
+-------------------------+------+----------+
|Servicio Público         |131838|28.87     |
|No aplica/No pertenece   |81027 |17.75     |
|Salud y Protección Social|76591 |16.77     |
|Educación Nacional       |23163 |5.07      |
|deportes                 |18433 |4.04      |
+-------------------------+------+----------+


 Top 5 para la columna: ORDEN
+--------------------+------+----------+
|orden               |count |porcentaje|
+--------------------+------+----------+
|Territorial         |330296|72.34     |
|Nacional            |121261|26.56     |
|Corporación Autónoma|5059  |1.11      |
|No Definido         |1     |0.0       |
+--------------------+------+----------+



In [27]:
# 2. ANÁLISIS TEMPORAL (Fechas y Duración)

print("\n Generando análisis temporal...")

# Calculamos la duración en días (Fecha Fin - Fecha Inicio)

df_tiempos = df_silver_typed.withColumn(
    "duracion_contrato_dias",
    F.datediff(F.col("fecha_de_fin_del_contrato"), F.col("fecha_de_inicio_del_contrato"))
)

# rango total de los datos
rango_temporal = df_tiempos.select(
    F.min("fecha_de_firma").alias("Primer_Contrato_Firmado"),
    F.max("fecha_de_firma").alias("Ultimo_Contrato_Firmado"),
    F.min("fecha_de_inicio_del_contrato").alias("Inicio_Ejecucion_Min"),
    F.max("fecha_de_fin_del_contrato").alias("Fin_Ejecucion_Max")
)

# Calculamos la duración promedio y mediana de los contratos
estadisticas_duracion = df_tiempos.select(
    F.round(F.mean("duracion_contrato_dias"), 2).alias("Duracion_Promedio_Dias"),
    F.expr("percentile_approx(duracion_contrato_dias, 0.5)").alias("Duracion_Mediana_Dias"),
    F.min("duracion_contrato_dias").alias("Duracion_Minima"),
    F.max("duracion_contrato_dias").alias("Duracion_Maxima")
)

# Mostrar resultados
print("\n=== RANGO CRONOLÓGICO DE LA DATA ===")
rango_temporal.show()

print("=== ESTADÍSTICAS DE DURACIÓN DE CONTRATOS ===")
estadisticas_duracion.show()


 Generando análisis temporal...

=== RANGO CRONOLÓGICO DE LA DATA ===
+-----------------------+-----------------------+--------------------+-------------------+
|Primer_Contrato_Firmado|Ultimo_Contrato_Firmado|Inicio_Ejecucion_Min|  Fin_Ejecucion_Max|
+-----------------------+-----------------------+--------------------+-------------------+
|    2025-07-01 00:00:00|    2025-12-31 00:00:00| 1899-11-06 00:00:00|5025-12-20 00:00:00|
+-----------------------+-----------------------+--------------------+-------------------+

=== ESTADÍSTICAS DE DURACIÓN DE CONTRATOS ===


[Stage 288:>                                                        (0 + 2) / 2]

+----------------------+---------------------+---------------+---------------+
|Duracion_Promedio_Dias|Duracion_Mediana_Dias|Duracion_Minima|Duracion_Maxima|
+----------------------+---------------------+---------------+---------------+
|                110.34|                   90|        -730477|        1095834|
+----------------------+---------------------+---------------+---------------+

